## Running the PBE Tool using the JSON File

### Getting the Saved Building Inventory and Ground Motion Records

In [48]:
#Import the pkl files in data folder
import pickle
from pathlib import Path

# Going to the folder where the Project is stored
shared_notebook_dir = Path(r'G:\Shared drives\\CS229 Project')
# notebook_dir = Path(r'C:\\Users\\varun\\OneDrive - Stanford\\Desktop\\Stanford\\5. Autumn 25 Quarter\\Independent Study CEE299')

# Going to the data folder inside the Project folder
with open(shared_notebook_dir / 'Code/data/filtered_buildings_Complete_with_unique_id.pkl', 'rb') as f:
    buildings_data = pickle.load(f)

print(f"Loaded {len(buildings_data)} buildings from complete data")

Loaded 2233792 buildings from complete data


In [3]:
### Import buildings_data_subset_chunk_4
import pandas as pd
# buildings_data_subset = pd.read_pickle(notebook_dir / 'Code/data/chunk4.pkl')
# buildings_data_subset.head(2)
buildings_data_subset_for_check = buildings_data.head(2)
buildings_data_subset_for_check.head()

,id,buildingheight,erabuilt,numstories,footprintArea,lon,lat,fparea,repaircost,constype,occupancy,fd_id,found_ht,ground_elv,geometry,unique_id
0,30,29,1967.0,1,9449.0,-117.662404,33.503555,9469.5,1628726.06,C1,COM4,477051287.0,0.5,125.264162,"POLYGON ((-117.66258 33.50378, -117.66246 33.5...",C1_COM4_0
1,31,30.800000000000001,1967.0,1,11196.0,-117.663316,33.503757,15480.0,2515049.75,W1,COM4,477051289.0,0.5,122.362539,"POLYGON ((-117.66359 33.50382, -117.66355 33.5...",W1_COM4_1


### Make a copy of the Template scInput JSON file

In [4]:
# make a copy of the scInput.json file with the filename having the unique_id 
# using the template file for each building in buildings_data
#"G:\Shared drives\CS229 Project\Code\data\JSON_Input_files\scInput_template.json"
import json
import shutil
from pathlib import Path
notebook_dir = Path(r'C:\\Users\\varun\\OneDrive - Stanford\\Desktop\\Stanford\\5. Autumn 25 Quarter\\Independent Study CEE299')
# Template file path
template_file = notebook_dir / "Code/JSON_Input_files/scInput_template.json"

# Output directory for individual building JSON files
output_dir = notebook_dir / "Code/JSON_Input_files/checking_local"
output_dir.mkdir(parents=True, exist_ok=True)

# Create a copy for each building with unique_id in filename
for idx, row in buildings_data_subset_for_check.iterrows(): ## RUNNING OVER BUILDING DATA SUBSET
    unique_id = row['unique_id']
    output_file = output_dir / f"scInput_{unique_id}.json"
    
    # Copy template
    shutil.copyfile(template_file, output_file)
    
    print(f"Created: {output_file.name}")

print(f"\nTotal files created: {len(buildings_data_subset_for_check)}")

Created: scInput_C1_COM4_0.json
Created: scInput_W1_COM4_1.json

Total files created: 2


### Make copies of the output folder template for each building simulation

In [5]:
# Make copies of the output folder template for each building with unique_id as folder name
import shutil
from pathlib import Path

# Template folder path
template_folder = notebook_dir / "PBE_Analysis/OUTPUT_FOLDER_TEMPLATE"

# Output directory for individual building folders
output_base_dir = notebook_dir / "PBE_Analysis/checking_local"
output_base_dir.mkdir(parents=True, exist_ok=True)

# Create a copy of the folder for each building with unique_id as folder name
for idx, row in buildings_data_subset_for_check.iterrows():
    unique_id = row['unique_id']
    output_folder = output_base_dir / unique_id
    
    # Copy entire folder structure
    if output_folder.exists():
        shutil.rmtree(output_folder)  # Remove if exists
    
    shutil.copytree(template_folder, output_folder)
    print(f"Created folder: {output_folder.name}")

print(f"\nTotal folders created: {len(buildings_data_subset_for_check)}")

Created folder: C1_COM4_0
Created folder: W1_COM4_1

Total folders created: 2


### Updating the input JSON files for each building in the Building Inventory Subset

In [43]:
# update the JSON input file to have only the buildings in buildings_data_subset 
import json
from sympy import sec
# Output directory for individual building JSON files
output_dir = notebook_dir / "Code/JSON_Input_files/checking_local"
# Output directory for individual building output folders
output_base_dir = notebook_dir / "PBE_Analysis/checking_local"
# for the input file created for the first building in buildings_data_subset later convert this to a loop for all buildings
JSON_INPUT_FILE = output_dir / f"scInput_{buildings_data_subset_for_check['unique_id'][1]}.json"
# for the output folder created for the first building in buildings_data_subset later convert this to a loop for all buildings
OUTPUT_FOLDER = output_base_dir / f"{buildings_data_subset_for_check['unique_id'][1]}"
with open(JSON_INPUT_FILE, 'r') as f:
    json_data = json.load(f)

# Update the building inventory path in the JSON data
json_data['DL']['Asset']['ComponentAssignmentFile'] = str(OUTPUT_FOLDER / "tmp.SimCenter/templatedir/CMP_QNT.csv") #CHECK IF NEED TO UPDATE EXCEL BEFORE EACH RUN,  Path to the CMP_QNT file in the OUTPUT_FOLDER
json_data['DL']["Asset"]["NumberOfStories"] = buildings_data_subset_for_check['numstories'][1]
json_data['DL']["Asset"]["OccupancyType"] = buildings_data_subset_for_check['occupancy'][1]
json_data['DL']["Asset"]["PlanArea"] = buildings_data_subset_for_check['fparea'][1] #footprintArea seems to have weird values, using fpArea instead (NSI?)
json_data["DL"]["Demands"]["SampleSize"] = "1000" #Earlier 10000 was giving 30s of run time per building, so reduced to 1000 for testing, should be fine 

# Events section corrected to proper indexing
"G:\Shared drives\CS229 Project\Code\Ground_motion_data\TimesSeries\RSN1731_NORTH392_SEA090.AT2"
# json_data["Events"][0]["Events"][0]["Records"][0]["fileName"]= "RSN1732_NORTH392_KAT000.AT2" 
# json_data["Events"][0]["Events"][0]["Records"][0]["fileName"]= "RSN1731_NORTH392_SEA090.AT2" 
json_data["Events"][0]["Events"][0]["Records"][0]["filePath"]= "G:/Shared drives/CS229 Project/Code/Ground_motion_data/TimesSeries"
# json_data["Events"][0]["Events"][0]["Records"][0]["fileName"]= "RSN1714_NORTH392_W70000.AT2" #TODO: Update this to the path of the ground motion file names in the new column
# json_data["Events"][0]["Events"][0]["Records"][0]["filePath"]= "C:/Users/varun/Downloads" #TODO: Update this to the path of the ground motion folder
json_data["Events"][0]["Events"][0]["name"] = "1731" #TODO Update this based on the column providing RSN number

json_data["GeneralInformation"]["DesignLevel"] = "High-Code" #Set to high for now TODO: change to middle/mid? check PBE Tool 

# Note : "units": {"force": "kips","length": "ft","temperature": "C","time": "sec"}
json_data["GeneralInformation"]["NumberOfStories"] = buildings_data_subset_for_check['numstories'][1]
json_data["GeneralInformation"]["PlanArea"] = buildings_data_subset_for_check['fparea'][1]
json_data["GeneralInformation"]["StructureType"] = buildings_data_subset_for_check['constype'][1]
json_data["GeneralInformation"]["name"] = buildings_data_subset_for_check['unique_id'][1]
json_data["GeneralInformation"]["YearBuilt"] = buildings_data_subset_for_check['erabuilt'][1]
json_data["GeneralInformation"]["height"] = buildings_data_subset_for_check['buildingheight'][1]
json_data["GeneralInformation"]["width"] = str(float((float(buildings_data_subset_for_check['fparea'][1])/float(buildings_data_subset_for_check['numstories'][1]))**0.5)) #Assuming square footprint for now
json_data["GeneralInformation"]["depth"] = str(float((float(buildings_data_subset_for_check['fparea'][1])/float(buildings_data_subset_for_check['numstories'][1]))**0.5)) #Assuming square footprint for now
json_data["GeneralInformation"]["location"]["latitude"] = buildings_data_subset_for_check['lat'][1]
json_data["GeneralInformation"]["location"]["longitude"] = buildings_data_subset_for_check['lon'][1]
json_data["GeneralInformation"]["stories"] = buildings_data_subset_for_check['numstories'][1]
json_data["GeneralInformation"]["planArea"] = buildings_data_subset_for_check['fparea'][1]


### SHOULD BE DIFFERENT FOR EACH PERSON !!!
json_data["localAppDir"] = "C:/Users/varun/Downloads/PBE_Windows_Download/PBE_Windows_Download"
json_data["remoteAppDir"] = "C:/Users/varun/Downloads/PBE_Windows_Download/PBE_Windows_Download"

## CHECK IF DIFFERENT. SHOULD BE FINE IF notebook_dir IS RIGHT
# json_data["runDir"] = "C:/Users/varun/OneDrive - Stanford/Desktop/Stanford/5. Autumn 25 Quarter/Independent Study CEE299/PBE_Analysis/From_GUI/tmp.SimCenter"
# json_data["workingDir"] = "C:/Users/varun/OneDrive - Stanford/Desktop/Stanford/5. Autumn 25 Quarter/Independent Study CEE299/PBE_Analysis/From_GUI"
json_data["runDir"] = str(OUTPUT_FOLDER / "tmp.SimCenter")
json_data["workingDir"] = str(OUTPUT_FOLDER)

# Update the JSON file
with open(JSON_INPUT_FILE, 'w') as f:
    json.dump(json_data, f, indent=4)


In [44]:
rec = json_data["Events"][0]["Events"][0]["Records"][0]
gm_full_path = Path(rec["filePath"]) / rec["fileName"]
print("GM path:", gm_full_path)
print("Exists:", gm_full_path.exists())

GM path: G:\Shared drives\CS229 Project\Code\Ground_motion_data\TimesSeries\RSN1714_NORTH392_W70000.AT2
Exists: True


### Running PBE Locally

In [45]:
import os
import subprocess
from pathlib import Path
import shutil

# Output directory for individual building JSON files
output_dir = notebook_dir / "Code/JSON_Input_files/checking_local"
# Output directory for individual building output folders
output_base_dir = notebook_dir / "PBE_Analysis/checking_local"
# for the input file created for the first building in buildings_data_subset later convert this to a loop for all buildings
JSON_INPUT_FILE = output_dir / f"scInput_{buildings_data_subset_for_check['unique_id'][1]}.json"
# for the output folder created for the first building in buildings_data_subset later convert this to a loop for all buildings
OUTPUT_FOLDER = output_base_dir / f"{buildings_data_subset_for_check['unique_id'][1]}"

# Make a copy of the JSON input file
CURRENT_SCINPUT_FILE = output_dir / "scInput.json"
shutil.copy(JSON_INPUT_FILE, CURRENT_SCINPUT_FILE)
### DIFFERENT FOR EACH PERSON !!!
WORKDIR = Path(r"C:\\Users\\varun\\Downloads") # WHERE YOUR PBE FILES ARE DOWNLOADED
PYTHON_EXE = Path(r"C:\\Users\\varun\\Downloads\\PBE_Windows_Download\\PBE_Windows_Download\\applications\\python\\python.exe")
PBE_WORKFLOW_SCRIPT = Path(r"C:\\Users\\varun\\Downloads\\PBE_Windows_Download\\PBE_Windows_Download\\applications\\Workflow\\sWHALE.py")
PBE_APP_WORKFLOW = Path(r"C:\\Users\\varun\\Downloads\\PBE_Windows_Download\\PBE_Windows_Download\\applications\\Workflow\\WorkflowApplications.json")

#Add Dakota bin folder to PATH for this run
env = os.environ.copy()
env["PATH"] = r"C:\\Users\\varun\\Downloads\\PBE_Windows_Download\\PBE_Windows_Download\\applications\\dakota\\bin" + os.pathsep + env["PATH"]
cmd = [
    str(PYTHON_EXE),
    str(PBE_WORKFLOW_SCRIPT),
    "runningLocal",
    str(CURRENT_SCINPUT_FILE),
    str(PBE_APP_WORKFLOW),
]

print("Running:\n", " ".join(f'"{c}"' for c in cmd))
print("CWD:", WORKDIR)

result = subprocess.run(
    cmd,
    cwd=str(WORKDIR),
    env=env,              # important: use modified PATH
    capture_output=True,
    text=True,
)

print("=== STDOUT ===")
print(result.stdout)
print("=== STDERR ===")
print(result.stderr)
print("Return code:", result.returncode)


Running:
 "C:\Users\varun\Downloads\PBE_Windows_Download\PBE_Windows_Download\applications\python\python.exe" "C:\Users\varun\Downloads\PBE_Windows_Download\PBE_Windows_Download\applications\Workflow\sWHALE.py" "runningLocal" "C:\Users\varun\OneDrive - Stanford\Desktop\Stanford\5. Autumn 25 Quarter\Independent Study CEE299\Code\JSON_Input_files\checking_local\scInput.json" "C:\Users\varun\Downloads\PBE_Windows_Download\PBE_Windows_Download\applications\Workflow\WorkflowApplications.json"
CWD: C:\Users\varun\Downloads
=== STDOUT ===

sWHALE workflow

System Information:
  local time zone: Pacific Standard Time
  start time: 2025-12-04T17:52:05
  python: 3.9.11 (tags/v3.9.11:2de452f, Mar 16 2022, 14:33:45) [MSC v.1929 64 bit (AMD64)]
  numpy: 1.26.4
  pandas: 2.2.3

--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
17:52:05 Started running the workflow script
         ---------

In [46]:
import json
from pathlib import Path

scinput_path = Path(r"C:\Users\varun\OneDrive - Stanford\Desktop\Stanford\5. Autumn 25 Quarter\Independent Study CEE299\Code\JSON_Input_files\checking_local\scInput.json")

with scinput_path.open("r", encoding="utf-8") as f:
    d = json.load(f)

rec = d["Events"][0]["Events"][0]["Records"][0]
print(rec)
from pathlib import Path

gm_full_path = Path(rec["filePath"]) / rec["fileName"]
print("GM path:", gm_full_path)
print("Exists:", gm_full_path.exists())

{'dirn': 1, 'factor': 1, 'fileName': 'RSN1714_NORTH392_W70000.AT2', 'filePath': 'G:/Shared drives/CS229 Project/Code/Ground_motion_data/TimesSeries'}
GM path: G:\Shared drives\CS229 Project\Code\Ground_motion_data\TimesSeries\RSN1714_NORTH392_W70000.AT2
Exists: True


### Running PBE Locally for the Files in GDrive

In [49]:
import pandas as pd
notebook_dir = shared_notebook_dir
# Output directory for individual building JSON files
output_dir = notebook_dir / "Code/JSON_Input_files/buildings"
buildings_data_subset = pd.read_pickle(notebook_dir / 'Code/data/chunk4.pkl')
buildings_data_subset.head(2)

,id,buildingheight,erabuilt,numstories,footprintArea,lon,lat,fparea,repaircost,constype,occupancy,fd_id,found_ht,ground_elv,geometry,unique_id,closest_record_sequence_number,closest_station_distance_km,file_name_horizontal_1,file_name_horizontal_2
2013,2551,8.9000000000000004,1993.0,1,2668.0,-118.621421,34.487599,2442.40000,9.656376e+04,H1,RES2,592199407.0,2.0,1210.919116,"POLYGON ((-118.62130 34.48766, -118.62130 34.4...",H1_RES2_2217402,1054,6.875308,NORTHR/PAR--L.at2,NORTHR/PAR--T.at2
2014,147,18.699999999999999,1969.0,1,16284.0,-118.542903,34.227775,8603.29518,1.154174e+06,S1,IND6,592113135.0,0.5,798.413210,"POLYGON ((-118.54302 34.22812, -118.54281 34.2...",S1_IND6_2126414,1048,3.167175,NORTHR/STC090.at2,NORTHR/STC180.at2


In [51]:
# make a copy of the scInput.json file with the filename having the unique_id 
# using the template file for each building in buildings_data
#"G:\Shared drives\CS229 Project\Code\data\JSON_Input_files\scInput_template.json"
import json
import shutil
from pathlib import Path
# notebook_dir = Path(r'C:\\Users\\varun\\OneDrive - Stanford\\Desktop\\Stanford\\5. Autumn 25 Quarter\\Independent Study CEE299')
# Template file path
template_file = notebook_dir / "Code/JSON_Input_files/scInput_template.json"

# Output directory for individual building JSON files
output_dir = notebook_dir / "Code/JSON_Input_files/buildings"
output_dir.mkdir(parents=True, exist_ok=True)

# Create a copy for each building with unique_id in filename
for idx, row in buildings_data_subset.iterrows(): ## RUNNING OVER BUILDING DATA SUBSET
    unique_id = row['unique_id']
    output_file = output_dir / f"scInput_{unique_id}.json"
    
    # Copy template
    shutil.copyfile(template_file, output_file)
    
    print(f"Created: {output_file.name}")

print(f"\nTotal files created: {len(buildings_data_subset)}")

Created: scInput_H1_RES2_2217402.json
Created: scInput_S1_IND6_2126414.json
Created: scInput_RM1_RES1_1919774.json
Created: scInput_W1_RES1_558374.json
Created: scInput_RM1_RES1_165793.json
Created: scInput_W1_COM4_1509412.json
Created: scInput_C1_RES3B_1013072.json
Created: scInput_RM1_RES1_45740.json
Created: scInput_S1_RES3B_163589.json
Created: scInput_W1_RES1_1646687.json
Created: scInput_W1_RES1_2189082.json
Created: scInput_W1_RES3C_865739.json
Created: scInput_W1_RES3A_844185.json
Created: scInput_S1_COM3_1115702.json
Created: scInput_RM1_RES1_1982564.json
Created: scInput_W1_RES1_1164736.json
Created: scInput_RM1_RES1_2194490.json
Created: scInput_W1_RES1_245893.json
Created: scInput_RM1_RES1_340189.json
Created: scInput_RM1_COM7_1467710.json
Created: scInput_RM1_RES1_2005076.json
Created: scInput_RM1_RES1_487005.json
Created: scInput_RM1_RES1_2119741.json
Created: scInput_RM1_COM3_1144234.json
Created: scInput_RM1_COM3_1506524.json
Created: scInput_W1_RES1_2066971.json
Create

In [52]:
# Make copies of the output folder template for each building with unique_id as folder name
import shutil
from pathlib import Path

# Template folder path
template_folder = notebook_dir / "PBE_Analysis/OUTPUT_FOLDER_TEMPLATE"

# Output directory for individual building folders
output_base_dir = notebook_dir / "PBE_Analysis/buildings/Chunk4"
output_base_dir.mkdir(parents=True, exist_ok=True)

# Create a copy of the folder for each building with unique_id as folder name
for idx, row in buildings_data_subset.iterrows():
    unique_id = row['unique_id']
    output_folder = output_base_dir / unique_id
    
    # Copy entire folder structure
    if output_folder.exists():
        shutil.rmtree(output_folder)  # Remove if exists
    
    shutil.copytree(template_folder, output_folder)
    print(f"Created folder: {output_folder.name}")

print(f"\nTotal folders created: {len(buildings_data_subset)}")

Created folder: H1_RES2_2217402
Created folder: S1_IND6_2126414
Created folder: RM1_RES1_1919774
Created folder: W1_RES1_558374
Created folder: RM1_RES1_165793
Created folder: W1_COM4_1509412
Created folder: C1_RES3B_1013072
Created folder: RM1_RES1_45740
Created folder: S1_RES3B_163589
Created folder: W1_RES1_1646687
Created folder: W1_RES1_2189082
Created folder: W1_RES3C_865739
Created folder: W1_RES3A_844185
Created folder: S1_COM3_1115702
Created folder: RM1_RES1_1982564
Created folder: W1_RES1_1164736
Created folder: RM1_RES1_2194490
Created folder: W1_RES1_245893
Created folder: RM1_RES1_340189
Created folder: RM1_COM7_1467710
Created folder: RM1_RES1_2005076
Created folder: RM1_RES1_487005
Created folder: RM1_RES1_2119741
Created folder: RM1_COM3_1144234
Created folder: RM1_COM3_1506524
Created folder: W1_RES1_2066971
Created folder: RM1_RES1_2074180
Created folder: W1_RES1_164287
Created folder: W1_RES1_1471085
Created folder: RM1_RES1_1338971
Created folder: S1_RES3A_862310
C

In [55]:
# update the JSON input file to have only the buildings in buildings_data_subset 
import json
from sympy import sec
# Output directory for individual building JSON files
output_dir = notebook_dir / "Code/JSON_Input_files/buildings"
# Output directory for individual building output folders
output_base_dir = notebook_dir / "PBE_Analysis/buildings/Chunk4"
# for the input file created for the first building in buildings_data_subset later convert this to a loop for all buildings
JSON_INPUT_FILE = output_dir / f"scInput_{buildings_data_subset_for_check['unique_id'][1]}.json"
# for the output folder created for the first building in buildings_data_subset later convert this to a loop for all buildings
OUTPUT_FOLDER = output_base_dir / f"{buildings_data_subset_for_check['unique_id'][1]}"
with open(JSON_INPUT_FILE, 'r') as f:
    json_data = json.load(f)

# Update the building inventory path in the JSON data
json_data['DL']['Asset']['ComponentAssignmentFile'] = str(OUTPUT_FOLDER / "tmp.SimCenter/templatedir/CMP_QNT.csv") #CHECK IF NEED TO UPDATE EXCEL BEFORE EACH RUN,  Path to the CMP_QNT file in the OUTPUT_FOLDER
json_data['DL']["Asset"]["NumberOfStories"] = buildings_data_subset_for_check['numstories'][1]
json_data['DL']["Asset"]["OccupancyType"] = buildings_data_subset_for_check['occupancy'][1]
json_data['DL']["Asset"]["PlanArea"] = buildings_data_subset_for_check['fparea'][1] #footprintArea seems to have weird values, using fpArea instead (NSI?)
json_data["DL"]["Demands"]["SampleSize"] = "1000" #Earlier 10000 was giving 30s of run time per building, so reduced to 1000 for testing, should be fine 

# Events section corrected to proper indexing
"G:\Shared drives\CS229 Project\Code\Ground_motion_data\TimesSeries\RSN1731_NORTH392_SEA090.AT2"
# json_data["Events"][0]["Events"][0]["Records"][0]["fileName"]= "RSN1732_NORTH392_KAT000.AT2" 
# json_data["Events"][0]["Events"][0]["Records"][0]["fileName"]= "RSN1731_NORTH392_SEA090.AT2" 
json_data["Events"][0]["Events"][0]["Records"][0]["filePath"]= "G:/Shared drives/CS229 Project/Code/Ground_motion_data/TimesSeries"
# json_data["Events"][0]["Events"][0]["Records"][0]["fileName"]= "RSN1714_NORTH392_W70000.AT2" #TODO: Update this to the path of the ground motion file names in the new column
# json_data["Events"][0]["Events"][0]["Records"][0]["filePath"]= "C:/Users/varun/Downloads" #TODO: Update this to the path of the ground motion folder
json_data["Events"][0]["Events"][0]["name"] = "1731" #TODO Update this based on the column providing RSN number

json_data["GeneralInformation"]["DesignLevel"] = "Moderate-Code" #Set to high for now TODO: change to middle/mid? check PBE Tool 

# Note : "units": {"force": "kips","length": "ft","temperature": "C","time": "sec"}
json_data["GeneralInformation"]["NumberOfStories"] = buildings_data_subset_for_check['numstories'][1]
json_data["GeneralInformation"]["PlanArea"] = buildings_data_subset_for_check['fparea'][1]
json_data["GeneralInformation"]["StructureType"] = buildings_data_subset_for_check['constype'][1]
json_data["GeneralInformation"]["name"] = buildings_data_subset_for_check['unique_id'][1]
json_data["GeneralInformation"]["YearBuilt"] = buildings_data_subset_for_check['erabuilt'][1]
json_data["GeneralInformation"]["height"] = buildings_data_subset_for_check['buildingheight'][1]
json_data["GeneralInformation"]["width"] = str(float((float(buildings_data_subset_for_check['fparea'][1])/float(buildings_data_subset_for_check['numstories'][1]))**0.5)) #Assuming square footprint for now
json_data["GeneralInformation"]["depth"] = str(float((float(buildings_data_subset_for_check['fparea'][1])/float(buildings_data_subset_for_check['numstories'][1]))**0.5)) #Assuming square footprint for now
json_data["GeneralInformation"]["location"]["latitude"] = buildings_data_subset_for_check['lat'][1]
json_data["GeneralInformation"]["location"]["longitude"] = buildings_data_subset_for_check['lon'][1]
json_data["GeneralInformation"]["stories"] = buildings_data_subset_for_check['numstories'][1]
json_data["GeneralInformation"]["planArea"] = buildings_data_subset_for_check['fparea'][1]


### SHOULD BE DIFFERENT FOR EACH PERSON !!!
json_data["localAppDir"] = "C:/Users/varun/Downloads/PBE_Windows_Download/PBE_Windows_Download"
json_data["remoteAppDir"] = "C:/Users/varun/Downloads/PBE_Windows_Download/PBE_Windows_Download"

## CHECK IF DIFFERENT. SHOULD BE FINE IF notebook_dir IS RIGHT
# json_data["runDir"] = "C:/Users/varun/OneDrive - Stanford/Desktop/Stanford/5. Autumn 25 Quarter/Independent Study CEE299/PBE_Analysis/From_GUI/tmp.SimCenter"
# json_data["workingDir"] = "C:/Users/varun/OneDrive - Stanford/Desktop/Stanford/5. Autumn 25 Quarter/Independent Study CEE299/PBE_Analysis/From_GUI"
json_data["runDir"] = str(OUTPUT_FOLDER / "tmp.SimCenter")
json_data["workingDir"] = str(OUTPUT_FOLDER)

# Update the JSON file
with open(JSON_INPUT_FILE, 'w') as f:
    json.dump(json_data, f, indent=4)

print("OK")


OK


In [56]:
rec = json_data["Events"][0]["Events"][0]["Records"][0]
gm_full_path = Path(rec["filePath"]) / rec["fileName"]
print("GM path:", gm_full_path)
print("Exists:", gm_full_path.exists())

GM path: G:\Shared drives\CS229 Project\Code\Ground_motion_data\TimesSeries\RSN1714_NORTH392_W70000.AT2
Exists: True


In [57]:
import os
import subprocess
from pathlib import Path
import shutil

# Output directory for individual building JSON files
output_dir = notebook_dir / "Code/JSON_Input_files/buildings"
# Output directory for individual building output folders
output_base_dir = notebook_dir / "PBE_Analysis/buildings/Chunk4"
# for the input file created for the first building in buildings_data_subset later convert this to a loop for all buildings
JSON_INPUT_FILE = output_dir / f"scInput_{buildings_data_subset_for_check['unique_id'][1]}.json"
# for the output folder created for the first building in buildings_data_subset later convert this to a loop for all buildings
OUTPUT_FOLDER = output_base_dir / f"{buildings_data_subset_for_check['unique_id'][1]}"

# Make a copy of the JSON input file
CURRENT_SCINPUT_FILE = output_dir / "scInput.json"
shutil.copy(JSON_INPUT_FILE, CURRENT_SCINPUT_FILE)
### DIFFERENT FOR EACH PERSON !!!
WORKDIR = Path(r"C:\\Users\\varun\\Downloads") # WHERE YOUR PBE FILES ARE DOWNLOADED
PYTHON_EXE = Path(r"C:\\Users\\varun\\Downloads\\PBE_Windows_Download\\PBE_Windows_Download\\applications\\python\\python.exe")
PBE_WORKFLOW_SCRIPT = Path(r"C:\\Users\\varun\\Downloads\\PBE_Windows_Download\\PBE_Windows_Download\\applications\\Workflow\\sWHALE.py")
PBE_APP_WORKFLOW = Path(r"C:\\Users\\varun\\Downloads\\PBE_Windows_Download\\PBE_Windows_Download\\applications\\Workflow\\WorkflowApplications.json")

#Add Dakota bin folder to PATH for this run
env = os.environ.copy()
env["PATH"] = r"C:\\Users\\varun\\Downloads\\PBE_Windows_Download\\PBE_Windows_Download\\applications\\dakota\\bin" + os.pathsep + env["PATH"]
cmd = [
    str(PYTHON_EXE),
    str(PBE_WORKFLOW_SCRIPT),
    "runningLocal",
    str(CURRENT_SCINPUT_FILE),
    str(PBE_APP_WORKFLOW),
]

print("Running:\n", " ".join(f'"{c}"' for c in cmd))
print("CWD:", WORKDIR)

result = subprocess.run(
    cmd,
    cwd=str(WORKDIR),
    env=env,              # important: use modified PATH
    capture_output=True,
    text=True,
)

print("=== STDOUT ===")
print(result.stdout)
print("=== STDERR ===")
print(result.stderr)
print("Return code:", result.returncode)


Running:
 "C:\Users\varun\Downloads\PBE_Windows_Download\PBE_Windows_Download\applications\python\python.exe" "C:\Users\varun\Downloads\PBE_Windows_Download\PBE_Windows_Download\applications\Workflow\sWHALE.py" "runningLocal" "G:\Shared drives\CS229 Project\Code\JSON_Input_files\buildings\scInput.json" "C:\Users\varun\Downloads\PBE_Windows_Download\PBE_Windows_Download\applications\Workflow\WorkflowApplications.json"
CWD: C:\Users\varun\Downloads
=== STDOUT ===

=== STDERR ===
Traceback (most recent call last):
  File "C:\Users\varun\Downloads\PBE_Windows_Download\PBE_Windows_Download\applications\Workflow\whale\main.py", line 174, in log_file
    with open(filepath, 'w', encoding='utf-8') as f:  # noqa: PTH123
FileNotFoundError: [Errno 2] No such file or directory: 'G:\\Shared drives\\CS229 Project\\PBE_Analysis\\buildings\\Chunk4\\W1_COM4_1\\tmp.SimCenter\\log.txt'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\User